# Lunar Lander: what makes an AI land well?

<a href="https://colab.research.google.com/github/ryeggouti/lunar-lander-rl/blob/main/lunar_lander.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Part 1 of my reinforcement learning project.** Part 2 is [Stork Flight](https://github.com/ryeggouti/stork-flight), where I built my own environment.

An AI flies a small spacecraft and has to land it between two flags. Nobody tells it how. After every move it gets a score: points for coming down gently and touching the pad, −100 for a crash, and a small cost each time it fires an engine. It learns, by trial and error, whatever earns the most.

**What I wanted to find out**
1. Can PPO, a standard learning algorithm, teach it to land?
2. What does its behaviour really depend on? I changed the rules three ways: free fuel, no crash penalty, and wind.
3. Are my results real, or just the luck of one training run?

## 1. Setup and a random pilot

First, a baseline: a pilot that presses random buttons. It shows how bad "knowing nothing" is, so I can tell how much the AI actually learns.

In [1]:
!pip install swig -q
!pip install "gymnasium[box2d]" stable-baselines3 imageio-ffmpeg -q
import gymnasium as gym
import numpy as np

env = gym.make("LunarLander-v3")
scores = []
for ep in range(20):
    obs, _ = env.reset(seed=ep)
    total, done = 0, False
    while not done:
        action = env.action_space.sample()
        obs, r, term, trunc, _ = env.step(action)
        total += r
        done = term or trunc
    scores.append(total)
print("Random pilot, average score:", round(np.mean(scores)))

Random pilot, average score: -209


**Result:** the random pilot averaged **−209**. Gymnasium counts **200 or more** as a solved landing, so that's the target.

## 2. Training

PPO from Stable-Baselines3, with 8 copies of the game running at once so it gets more practice per minute. The settings are the ones commonly used for Lunar Lander in the RL Baselines3 Zoo. I trained for 1 million steps and saved a snapshot every 100,000, so I could watch it improve later.

In [2]:
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.callbacks import BaseCallback, CheckpointCallback

class Progress(BaseCallback):
    # prints the recent average score every 100,000 steps
    def _on_step(self):
        if self.num_timesteps % 100_000 < self.training_env.num_envs:
            buf = [e["r"] for e in self.model.ep_info_buffer]
            score = round(np.mean(buf)) if buf else "-"
            print(self.num_timesteps, "steps, recent score:", score)
        return True

venv = make_vec_env("LunarLander-v3", n_envs=8, seed=0)
model = PPO("MlpPolicy", venv, n_steps=1024, batch_size=64,
            n_epochs=4, gamma=0.999, gae_lambda=0.98,
            ent_coef=0.01, seed=0, verbose=0)
ckpt = CheckpointCallback(save_freq=12_500, save_path="checkpoints",
                          name_prefix="lander")
model.learn(total_timesteps=1_000_000, callback=[Progress(), ckpt])
model.save("lander_final")
print("Training done")

100000 steps, recent score: -44


200000 steps, recent score: 3
300000 steps, recent score: 64
400000 steps, recent score: 99
500000 steps, recent score: 120
600000 steps, recent score: 115
700000 steps, recent score: 129
800000 steps, recent score: 125
900000 steps, recent score: 136
1000000 steps, recent score: 136
Training done


**Result:** the recent score climbed from **−44** at 100k steps to **136** at 1 million. Most of the progress happened before 500k; after that it improved slowly.

## 3. Testing the trained pilot

Training scores include random exploration, so I tested the finished pilot separately on 20 new landings, always choosing its best move. Then I recorded one landing for each pilot.

In [3]:
import imageio
from IPython.display import Video
from stable_baselines3.common.evaluation import evaluate_policy

test_env = gym.make("LunarLander-v3")
mean, std = evaluate_policy(model, test_env, n_eval_episodes=20)
print(f"Trained pilot: {mean:.0f} average score (± {std:.0f})")

def record(m, path, seed=1):
    env = gym.make("LunarLander-v3", render_mode="rgb_array")
    obs, _ = env.reset(seed=seed)
    frames, done, total = [env.render()], False, 0
    while not done:
        if m is None:
            action = env.action_space.sample()
        else:
            action, _ = m.predict(obs, deterministic=True)
        obs, r, term, trunc, _ = env.step(action)
        total += r
        done = term or trunc
        frames.append(env.render())
    imageio.mimsave(path, frames, fps=30)
    return round(total)

print("Random:", record(None, "random.mp4"))
print("Trained:", record(model, "trained.mp4"))
Video("trained.mp4", embed=True)

Trained pilot: 237 average score (± 37)


Random: -79


Trained: 242


**Result:** **237 ± 37** on average, above the 200 "solved" line. In the recorded flight, the trained pilot scored **242** and the random one **−79**.

Saving the trained model and the two videos:

In [4]:
from google.colab import files
files.download("lander_final.zip")
files.download("trained.mp4")
files.download("random.mp4")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 4. Watching it learn

The snapshots from training let me test the same AI at different amounts of practice, on 10 landings each, and record a video of each one.

In [5]:
progress = []
for steps in [100_000, 300_000, 500_000, 1_000_000]:
    path = f"checkpoints/lander_{steps}_steps.zip"
    m = PPO.load(path)
    mean, _ = evaluate_policy(m, test_env, n_eval_episodes=10)
    score = record(m, f"lander_{steps // 1000}k.mp4")
    progress.append((steps, round(mean), score))
    print(f"{steps:>9,} steps: test average {mean:.0f}, video run {score}")

for steps, _, _ in progress:
    files.download(f"lander_{steps // 1000}k.mp4")

  100,000 steps: test average -163, video run -233


  300,000 steps: test average 3, video run 174


  500,000 steps: test average 150, video run 200


1,000,000 steps: test average 261, video run 239


<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**Result:**

| Practice | Average score |
|---|---|
| 100k steps | −163 |
| 300k steps | 3 |
| 500k steps | 150 |
| 1 million steps | 261 |

At 100k it mostly falls or drifts away. By 500k it lands, but slowly. By 1 million it lands cleanly.

## 5. Changing the rules

If the AI only learns from its score, then changing what earns points should change how it flies. I changed one rule at a time:
- **Free fuel:** firing the engines costs nothing.
- **No crash penalty:** a crash costs 0 instead of −100.
- **Wind:** a strong, gusty wind pushes the lander around.

To compare fairly, every AI is tested on the same 20 starting positions and always scored with the **normal** rules, even if it trained with different ones. I also count how it lands: crashes, time-outs, seconds per landing and engine burns.

In [6]:
import pandas as pd

class FreeFuel(gym.Wrapper):
    # gives back the fuel cost, so firing engines is free
    def step(self, a):
        obs, r, term, trunc, info = self.env.step(a)
        if a == 2:
            r += 0.3
        elif a in (1, 3):
            r += 0.03
        return obs, r, term, trunc, info

class NoCrashPenalty(gym.Wrapper):
    # a crash costs 0 instead of -100
    def step(self, a):
        obs, r, term, trunc, info = self.env.step(a)
        if term and r == -100:
            r = 0
        return obs, r, term, trunc, info

WIND = {"enable_wind": True, "wind_power": 18.0, "turbulence_power": 1.5}

def measure(m, n=20, **kw):
    # 20 landings, same starts for every AI, scored on the normal rules
    env = gym.make("LunarLander-v3", **kw)
    rows = []
    for ep in range(n):
        obs, _ = env.reset(seed=1000 + ep)
        total, steps, main, side, done = 0, 0, 0, 0, False
        while not done:
            a, _ = m.predict(obs, deterministic=True)
            a = int(a)
            main += a == 2
            side += a in (1, 3)
            obs, r, term, trunc, _ = env.step(a)
            total += r
            steps += 1
            done = term or trunc
        rows.append({"score": total, "landed": term and r == 100,
                     "crashed": term and r == -100, "timeout": trunc,
                     "seconds": steps / 50, "main_fires": main,
                     "side_fires": side})
    return pd.DataFrame(rows)

def summary(df):
    return {"score": round(df.score.mean()),
            "landed %": round(df.landed.mean() * 100),
            "crashed %": round(df.crashed.mean() * 100),
            "timeout %": round(df.timeout.mean() * 100),
            "seconds": round(df.seconds.mean(), 1),
            "main fires": round(df.main_fires.mean()),
            "side fires": round(df.side_fires.mean())}
print("Ready")

Ready


Each experiment trains a new AI for 500k steps (the same as the normal AI's 500k snapshot):

In [7]:
experiments = {
    "free_fuel": {"wrapper_class": FreeFuel},
    "no_crash_penalty": {"wrapper_class": NoCrashPenalty},
    "wind": {"env_kwargs": WIND},
}
models = {}
for name, opts in experiments.items():
    print("Training:", name)
    venv = make_vec_env("LunarLander-v3", n_envs=8, seed=0, **opts)
    m = PPO("MlpPolicy", venv, n_steps=1024, batch_size=64,
            n_epochs=4, gamma=0.999, gae_lambda=0.98,
            ent_coef=0.01, seed=0, verbose=0)
    m.learn(total_timesteps=500_000, callback=Progress())
    m.save(f"exp_{name}")
    models[name] = m
print("All experiments done")

Training: free_fuel
100000 steps, recent score: -50
200000 steps, recent score: -19
300000 steps, recent score: 69
400000 steps, recent score: 95
500000 steps, recent score: 95
Training: no_crash_penalty
100000 steps, recent score: -14
200000 steps, recent score: 39
300000 steps, recent score: 77
400000 steps, recent score: 120
500000 steps, recent score: 151
Training: wind
100000 steps, recent score: -113
200000 steps, recent score: -58
300000 steps, recent score: -21
400000 steps, recent score: 12
500000 steps, recent score: 24
All experiments done


Testing every AI on the same 20 starts, and recording one landing from each experiment:

In [8]:
base500 = PPO.load("checkpoints/lander_500000_steps.zip")
rows = {
    "normal (500k)": summary(measure(base500)),
    "normal (1M)": summary(measure(model)),
    "free fuel (500k)": summary(measure(models["free_fuel"])),
    "no crash penalty (500k)": summary(measure(models["no_crash_penalty"])),
    "normal 1M AI, in wind": summary(measure(model, **WIND)),
    "wind-trained (500k), in wind": summary(measure(models["wind"], **WIND)),
}
results = pd.DataFrame(rows).T
print(results.to_string())
results.to_csv("experiments.csv")

def record_kw(m, path, seed=1, **kw):
    env = gym.make("LunarLander-v3", render_mode="rgb_array", **kw)
    obs, _ = env.reset(seed=seed)
    frames, done = [env.render()], False
    while not done:
        a, _ = m.predict(obs, deterministic=True)
        obs, r, term, trunc, _ = env.step(a)
        done = term or trunc
        frames.append(env.render())
    imageio.mimsave(path, frames, fps=30)

record_kw(models["free_fuel"], "exp_free_fuel.mp4")
record_kw(models["no_crash_penalty"], "exp_no_crash.mp4")
record_kw(model, "exp_normal_in_wind.mp4", **WIND)
record_kw(models["wind"], "exp_wind_trained.mp4", **WIND)
for f in ["experiments.csv", "exp_free_fuel.mp4", "exp_no_crash.mp4",
          "exp_normal_in_wind.mp4", "exp_wind_trained.mp4"]:
    files.download(f)

                              score  landed %  crashed %  timeout %  seconds  main fires  side fires
normal (500k)                 176.0      85.0        5.0       10.0     12.9       324.0       268.0
normal (1M)                   255.0     100.0        0.0        0.0      7.1       171.0       136.0
free fuel (500k)               86.0      55.0        0.0       45.0     18.4       494.0       328.0
no crash penalty (500k)       271.0     100.0        0.0        0.0      5.5       127.0        27.0
normal 1M AI, in wind          23.0      20.0       80.0        0.0      6.9       168.0       168.0
wind-trained (500k), in wind   12.0      40.0       35.0       25.0     15.5       394.0       272.0


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**Result:**
- **Free fuel made it worse** (86 vs 176). With no cost for the engines, it fired the main engine about 50% more and hovered until time ran out in 45% of the tests.
- **No crash penalty made it better**, which surprised me: 271, it landed every time, in 5.5 seconds instead of 12.9, and used its side engines far less (27 burns instead of 268). It even beat the normal AI that trained twice as long (255).
- **Wind broke it.** The AI that trained without wind crashed 80% of the time in wind. The wind-trained AI crashed less (35%) but scored only 12, and it had half the practice, so that comparison isn't fully fair.

## 6. Was it luck?

The crash-penalty result was the surprising one, and one training run can be lucky. So I trained both the normal and the no-crash-penalty versions again with two more random seeds and tested them the same way.

In [9]:
seed_rows = [
    {"setup": "normal", "seed": 0, **summary(measure(base500))},
    {"setup": "no crash penalty", "seed": 0,
     **summary(measure(models["no_crash_penalty"]))},
]
setups = {"normal": {}, "no crash penalty": {"wrapper_class": NoCrashPenalty}}

for seed in [1, 2]:
    for name, opts in setups.items():
        print(f"Training: {name}, seed {seed}")
        venv = make_vec_env("LunarLander-v3", n_envs=8, seed=seed, **opts)
        m = PPO("MlpPolicy", venv, n_steps=1024, batch_size=64,
                n_epochs=4, gamma=0.999, gae_lambda=0.98,
                ent_coef=0.01, seed=seed, verbose=0)
        m.learn(total_timesteps=500_000, callback=Progress())
        m.save(f"seed{seed}_{name.replace(' ', '_')}")
        seed_rows.append({"setup": name, "seed": seed, **summary(measure(m))})

seeds = pd.DataFrame(seed_rows)
print(seeds.to_string(index=False))
print("\nAverage over 3 seeds:")
print(seeds.groupby("setup")[["score", "landed %", "crashed %", "seconds"]]
      .agg(["mean", "min", "max"]).round(1).to_string())
seeds.to_csv("seeds.csv")
files.download("seeds.csv")

Training: normal, seed 1
100000 steps, recent score: -47
200000 steps, recent score: 17
300000 steps, recent score: 65
400000 steps, recent score: 107
500000 steps, recent score: 116
Training: no crash penalty, seed 1
100000 steps, recent score: -31
200000 steps, recent score: 37
300000 steps, recent score: 51
400000 steps, recent score: 227
500000 steps, recent score: 262
Training: normal, seed 2
100000 steps, recent score: -59
200000 steps, recent score: -11
300000 steps, recent score: 72
400000 steps, recent score: 97
500000 steps, recent score: 91
Training: no crash penalty, seed 2
100000 steps, recent score: -16
200000 steps, recent score: 50
300000 steps, recent score: 88
400000 steps, recent score: 101
500000 steps, recent score: 157
           setup  seed  score  landed %  crashed %  timeout %  seconds  main fires  side fires
          normal     0    176        85          5         10     12.9         324         268
no crash penalty     0    271       100          0         

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**Result:** it held every time. Without the crash penalty, scores were **271, 256 and 262** and it landed 100% of the time in all three runs. With the normal rules, scores were **176, 192 and 196**, landing 60–85% of the time.

Backing up the models and results, in case Colab disconnects:

In [10]:
import shutil, glob
import os
os.makedirs("backup", exist_ok=True)
for f in glob.glob("*.zip") + glob.glob("*.csv"):
    shutil.copy(f, "backup")
shutil.copytree("checkpoints", "backup/checkpoints", dirs_exist_ok=True)
shutil.make_archive("lunar_lander_backup", "zip", "backup")
files.download("lunar_lander_backup.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## What I found

1. **PPO learns to land:** from −209 (random) to 261 after 1 million steps.
2. **The rules shape the behaviour more than I expected.** Free fuel made it lazy and slow. Removing the crash penalty made it faster and more reliable, and that held across 3 training runs.
3. **My best explanation** (a hypothesis, not something I proved): a −100 crash penalty teaches the AI that getting close to the ground is dangerous, so it plays it safe by hovering and correcting with its side engines, and sometimes runs out of time. Without the penalty it still wants the +100 for landing, so it commits and lands.
4. **It doesn't adapt to conditions it never practised in:** wind it had never felt made it crash 80% of the time.

## Limits
- 20 test landings per AI is a small sample.
- Only the crash-penalty experiment was repeated with 3 seeds. Free fuel and wind were trained once.
- The wind comparison mixes different amounts of training (1M vs 500k steps).

I wrote this project with help from an AI coding assistant. The questions, the experiments and the conclusions are mine.